# Identification of Somatic Variants in Breast Cancer Whole Exome Sequencing (WES) Data

## 1. TOOLS INSTALLATION

In [21]:
import os, glob
import pandas as pd

# Command-line tools + Java 17 (for GATK4)
!apt-get -qq update
!apt-get -qq install -y sra-toolkit fastqc bwa samtools bcftools tabix  openjdk-17-jdk-headless

# Python QC & Trimming tools
!pip -q install cutadapt multiqc

# GATK4 for Variant calling
GATK_VER = '4.5.0.0'
if not os.path.isdir(f'gatk-{GATK_VER}'):
    os.system(f'wget \
              -q https://github.com/broadinstitute/gatk/releases/download/{GATK_VER}/gatk-{GATK_VER}.zip')
    os.system(f'unzip -q gatk-{GATK_VER}.zip')
os.environ['PATH'] = os.path.abspath(f'gatk-{GATK_VER}') + ':' + os.environ['PATH']
!gatk --version

# snpEff for annotation
!wget -O snpEff_latest_core.zip "https://sourceforge.net/projects/snpeff/files/snpEff_latest_core.zip/download" 
!unzip -o snpEff_latest_core.zip

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Using GATK jar /content/gatk-4.5.0.0/gatk-package-4.5.0.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /content/gatk-4.5.0.0/gatk-package-4.5.0.0-local.jar --version
The Genome Analysis Toolkit (GATK) v4.5.0.0
HTSJDK Version: 4.1.0
Picard Version: 3.1.1
--2026-07-01 07:21:01--  https://sourceforge.net/projects/snpeff/files/snpEff_latest_core.zip/download
Resolving sourceforge.net (sourceforge.net)... 104.18.13.149, 104.18.12.149, 2606:4700::6812:c95, ...
Connecting to sourceforge.net (sourceforge.net)|104.18.13.149|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://downloads.sourceforge.net/project/snpeff/snpEff_latest_core.

## 2. DATA LOADING

In [4]:
# Fetching the SRP676624 run list (run accession + sample title + FASTQ links) from the ENA.
!wget -q -O srp676624_runs.tsv \
  "https://www.ebi.ac.uk/ena/portal/api/filereport?accession=SRP676624&result=read_run&fields=run_accession,sample_accession,sample_title,library_strategy,fastq_ftp,fastq_bytes&format=tsv"
runs = pd.read_csv('srp676624_runs.tsv', sep='\t')
runs.head(50)

,run_accession,sample_accession,sample_title,library_strategy,fastq_ftp,fastq_bytes
0,SRR37211293,SAMN55324210,NaN,WXS,ftp.sra.ebi.ac.uk/vol1/fastq/SRR372/093/SRR372...,2579290261;2688533468
1,SRR37211298,SAMN55324243,NaN,WXS,ftp.sra.ebi.ac.uk/vol1/fastq/SRR372/098/SRR372...,1854458159;1915424571
2,SRR37211300,SAMN55324241,NaN,WXS,ftp.sra.ebi.ac.uk/vol1/fastq/SRR372/000/SRR372...,1841449725;1895408325
3,SRR37211301,SAMN55324205,NaN,WXS,ftp.sra.ebi.ac.uk/vol1/fastq/SRR372/001/SRR372...,2498625858;2578106280
4,SRR37211302,SAMN55324240,NaN,WXS,ftp.sra.ebi.ac.uk/vol1/fastq/SRR372/002/SRR372...,1814476777;1886141129
5,SRR37211303,SAMN55324239,NaN,WXS,ftp.sra.ebi.ac.uk/vol1/fastq/SRR372/003/SRR372...,1960501589;2004429244
6,SRR37211304,SAMN55324238,NaN,WXS,ftp.sra.ebi.ac.uk/vol1/fastq/SRR372/004/SRR372...,1710615692;1769195977
7,SRR37211306,SAMN55324236,NaN,WXS,ftp.sra.ebi.ac.uk/vol1/fastq/SRR372/006/SRR372...,1802012768;1847679940
8,SRR37211309,SAMN55324233,NaN,WXS,ftp.sra.ebi.ac.uk/vol1/fastq/SRR372/009/SRR372...,1585094193;1625187862
9,SRR37211311,SAMN55324231,NaN,WXS,ftp.sra.ebi.ac.uk/vol1/fastq/SRR372/011/SRR372...,1525616678;1609243356


In [22]:
# Configuration for the analysis

# Samples
SAMPLE_SHEET = [
    ("SRR37211323", "Sample_1"),
    ("SRR37211334", "Sample_2"),
    ("SRR37211335", "Sample_3")
]

# Reference subset
CHROMS  = ['chr13', 'chr17'] 
THREADS = 4

# Directory layout
REF_DIR, RES_DIR, FASTQ_DIR, QC_DIR, QC_POST = 'reference', 'resources', 'fastq', 'qc', 'qc_post'
TRIM_DIR, BAM_DIR, VCF_DIR                   = 'trimmed', 'bam', 'vcf'
ANN_DIR, RESULTS_DIR                         = 'annotation', 'results'
for d in [REF_DIR,RES_DIR,FASTQ_DIR,QC_DIR,QC_POST,TRIM_DIR,BAM_DIR,VCF_DIR,ANN_DIR,RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

REF_FA   = os.path.join(REF_DIR, 'GRCh38_subset.fa')
GERMLINE = os.path.join(RES_DIR, 'af-only-gnomad.subset.vcf.gz')
PON      = os.path.join(RES_DIR, '1000g_pon.subset.vcf.gz')
COMMON   = os.path.join(RES_DIR, 'small_exac_common.subset.vcf.gz')
INTERVALS = ' '.join(f'-L {c}' for c in CHROMS)

print(f'{len(SAMPLE_SHEET)} samples | reference: {", ".join(CHROMS)}')

3 samples | reference: chr13, chr17


In [4]:
# Reference genome subset (GRCh38) for the analysis
open(REF_FA, 'w').close()
for chrom in CHROMS:
    gz = os.path.join(REF_DIR, f'{chrom}.fa.gz')
    if not os.path.exists(gz):
        os.system(f'wget -q https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/{chrom}.fa.gz -O {gz}')
    os.system(f'gunzip -kf {gz}')
    os.system(f'cat {os.path.join(REF_DIR, chrom + ".fa")} >> {REF_FA}')
!bwa index {REF_FA}
!samtools faidx {REF_FA}
!gatk CreateSequenceDictionary -R {REF_FA}
print('Reference built and indexed:', REF_FA)

[bwa_index] Pack FASTA... 3.04 sec
[bwa_index] Construct BWT for the packed sequence...
[BWTIncCreate] textLength=395243538, availableWord=39810548
[BWTIncConstructFromPacked] 10 iterations done. 65669826 characters processed.
[BWTIncConstructFromPacked] 20 iterations done. 121321010 characters processed.
[BWTIncConstructFromPacked] 30 iterations done. 170779906 characters processed.
[BWTIncConstructFromPacked] 40 iterations done. 214735074 characters processed.
[BWTIncConstructFromPacked] 50 iterations done. 253798514 characters processed.
[BWTIncConstructFromPacked] 60 iterations done. 288514210 characters processed.
[BWTIncConstructFromPacked] 70 iterations done. 319365538 characters processed.
[BWTIncConstructFromPacked] 80 iterations done. 346782242 characters processed.
[BWTIncConstructFromPacked] 90 iterations done. 371146258 characters processed.
[BWTIncConstructFromPacked] 100 iterations done. 392797010 characters processed.
[bwt_gen] Finished constructing BWT in 102 iteration

In [5]:
# GATK4 resources subset for the analysis
BASE = 'https://storage.googleapis.com/gatk-best-practices/somatic-hg38'
downloads = {
    'af-only-gnomad.hg38.vcf.gz'   : GERMLINE,
    '1000g_pon.hg38.vcf.gz'        : PON,
    'small_exac_common_3.hg38.vcf.gz': COMMON,
}
for fname, subset_out in downloads.items():
    raw = os.path.join(RES_DIR, fname)
    if not os.path.exists(raw):
        print('Downloading', fname)
        os.system(f'wget -q {BASE}/{fname} -O {raw}')
        os.system(f'wget -q {BASE}/{fname}.tbi -O {raw}.tbi')
    os.system(f'gatk SelectVariants -R {REF_FA} -V {raw} {INTERVALS} -O {subset_out}')
print('Resources ready.')

Resources ready.


In [8]:
# Downloading FASTQ files from SRA
for run, _ in SAMPLE_SHEET:
    if os.path.exists(os.path.join(FASTQ_DIR, f'{run}_1.fastq')):continue
    print('Downloading', run)
    os.system(f'prefetch {run} -O {FASTQ_DIR}')
    os.system(f'fasterq-dump {FASTQ_DIR}/{run}/{run}.sra --split-files -e {THREADS} -O {FASTQ_DIR}')

## 3. Quality Control & Adapter Trimming

In [10]:
# Implementing initial QC with FastQC and MultiQC
fastqs = sorted(glob.glob(os.path.join(FASTQ_DIR, '*.fastq')))
for fq in fastqs:
    print('Running FastQC on', fq)
    os.system(f'fastqc -q -t {THREADS} -o {QC_DIR} {fq}')
!multiqc -q -f -o qc qc

Running FastQC on fastq/SRR37211323_1.fastq
Running FastQC on fastq/SRR37211323_2.fastq
Running FastQC on fastq/SRR37211334_1.fastq
Running FastQC on fastq/SRR37211334_2.fastq
Running FastQC on fastq/SRR37211335_1.fastq
Running FastQC on fastq/SRR37211335_2.fastq


In [11]:
# Implementing Adapter trimming with Cutadapt
FWD = REV = 'AGATCGGAAGAGC'
for run, _ in SAMPLE_SHEET:
    in1 = os.path.join(FASTQ_DIR, f'{run}_1.fastq')
    in2 = os.path.join(FASTQ_DIR, f'{run}_2.fastq')
    o1  = os.path.join(TRIM_DIR, f'{run}_1.trim.fastq.gz')
    o2  = os.path.join(TRIM_DIR, f'{run}_2.trim.fastq.gz')
    print('Trimming', run)
    os.system(f'cutadapt -j {THREADS} -a {FWD} -A {REV} -q 20 -m 30 -o {o1} -p {o2} {in1} {in2}')

Trimming SRR37211323
Trimming SRR37211334
Trimming SRR37211335


In [12]:
# Implementing Post-Trimming QC with FastQC and MultiQC
fastqs = sorted(glob.glob(os.path.join(TRIM_DIR, '*.trim.fastq.gz')))
for fq in fastqs:
    print('Running FastQC on', fq)
    os.system(f'fastqc -q -t {THREADS} -o {QC_POST} {fq}')
!multiqc -q -f -o qc_post qc_post

Running FastQC on trimmed/SRR37211323_1.trim.fastq.gz
Running FastQC on trimmed/SRR37211323_2.trim.fastq.gz
Running FastQC on trimmed/SRR37211334_1.trim.fastq.gz
Running FastQC on trimmed/SRR37211334_2.trim.fastq.gz
Running FastQC on trimmed/SRR37211335_1.trim.fastq.gz
Running FastQC on trimmed/SRR37211335_2.trim.fastq.gz


## 4. Read Alignment & Post-alignment Processing

In [ ]:
# Aligning reads to the reference genome with BWA-MEM, sorting and marking duplicates with GATK4
def process_sample(run, name):
    bs = chr(92)
    rg = f'@RG{bs}tID:{name}{bs}tSM:{name}{bs}tPL:ILLUMINA{bs}tLB:{name}'
    r1 = os.path.join(TRIM_DIR, f'{run}_1.trim.fastq.gz')
    r2 = os.path.join(TRIM_DIR, f'{run}_2.trim.fastq.gz')
    sort_bam = os.path.join(BAM_DIR, f'{name}.sorted.bam')
    dedup    = os.path.join(BAM_DIR, f'{name}.dedup.bam')
    metrics  = os.path.join(BAM_DIR, f'{name}.dup_metrics.txt')
    print(f'  aligning {name}')
    os.system(f"bwa mem -t {THREADS} -R '{rg}' {REF_FA} {r1} {r2} "
              f"| samtools sort -@ {THREADS} -o {sort_bam} -")
    os.system(f'samtools index {sort_bam}')
    print(f'  marking duplicates {name}')
    os.system(f'gatk MarkDuplicates -I {sort_bam} -O {dedup} -M {metrics}')
    os.system(f'samtools index {dedup}')
    return dedup
	
bam_paths = {}
for run, name in SAMPLE_SHEET:
    print('Processing', name)
    bam_paths[name] = process_sample(run, name)

Processing Sample_1
  aligning Sample_1
  marking duplicates Sample_1
Processing Sample_2
  aligning Sample_2
  marking duplicates Sample_2
Processing Sample_3
  aligning Sample_3
  marking duplicates Sample_3


In [9]:
# Downloading and subsetting known-sites for BQSR (Base Quality Score Recalibration)
BROAD = 'https://storage.googleapis.com/gcp-public-data--broad-references/hg38/v0'

INCLUDE_DBSNP = True

ks_files = {
    'Mills_and_1000G_gold_standard.indels.hg38.vcf.gz': ('.tbi', os.path.join(RES_DIR, 'mills.subset.vcf.gz')),
    'Homo_sapiens_assembly38.known_indels.vcf.gz'      : ('.tbi', os.path.join(RES_DIR, 'known_indels.subset.vcf.gz')),
}
if INCLUDE_DBSNP:
    ks_files['Homo_sapiens_assembly38.dbsnp138.vcf'] = ('.idx', os.path.join(RES_DIR, 'dbsnp.subset.vcf.gz'))

KNOWN_SITES = []
for fname, (idx_suffix, subset_out) in ks_files.items():
    raw = os.path.join(RES_DIR, fname)
    if not os.path.exists(raw):
        big = ' (large — several GB)' if 'dbsnp' in fname else ''
        print('Downloading', fname, big)
        os.system(f'wget -q {BROAD}/{fname} -O {raw}')
        os.system(f'wget -q {BROAD}/{fname}{idx_suffix} -O {raw}{idx_suffix}')
    print('Subsetting', fname, '-> chr13, chr17')
    os.system(f'gatk SelectVariants -R {REF_FA} -V {raw} {INTERVALS} -O {subset_out}')
    KNOWN_SITES.append(subset_out)
print('Known-sites ready:', KNOWN_SITES)

Subsetting Mills_and_1000G_gold_standard.indels.hg38.vcf.gz -> chr13, chr17
Subsetting Homo_sapiens_assembly38.known_indels.vcf.gz -> chr13, chr17
Subsetting Homo_sapiens_assembly38.dbsnp138.vcf -> chr13, chr17
Known-sites ready: ['resources/mills.subset.vcf.gz', 'resources/known_indels.subset.vcf.gz', 'resources/dbsnp.subset.vcf.gz']


In [ ]:
# Running BQSR per sample
ks_args = ' '.join(f'--known-sites {k}' for k in KNOWN_SITES)
for run, name in SAMPLE_SHEET:
    dedup = os.path.join(BAM_DIR, f'{name}.dedup.bam')
    table = os.path.join(BAM_DIR, f'{name}.recal.table')
    recal = os.path.join(BAM_DIR, f'{name}.recal.bam')
    print('BQSR for', name)
    os.system(f'gatk BaseRecalibrator -R {REF_FA} -I {dedup} {ks_args} {INTERVALS} -O {table}')
    os.system(f'gatk ApplyBQSR -R {REF_FA} -I {dedup} --bqsr-recal-file {table} {INTERVALS} -O {recal}')
    os.system(f'samtools index {recal}')
    bam_paths[name] = recal

BQSR for Sample_1
BQSR for Sample_2
BQSR for Sample_3


In [15]:
# Alignment summary per sample
for run, name in SAMPLE_SHEET:
    recal = os.path.join(BAM_DIR, f'{name}.recal.bam')
    print('====', name, '====')
    os.system(f'samtools flagstat {recal}')
    print(os.popen(f'samtools flagstat {recal}').read())

==== Sample_1 ====
7596716 + 0 in total (QC-passed reads + QC-failed reads)
7542950 + 0 primary
0 + 0 secondary
53766 + 0 supplementary
627218 + 0 duplicates
627218 + 0 primary duplicates
7094566 + 0 mapped (93.39% : N/A)
7040800 + 0 primary mapped (93.34% : N/A)
7542950 + 0 paired in sequencing
3771475 + 0 read1
3771475 + 0 read2
6343396 + 0 properly paired (84.10% : N/A)
6538650 + 0 with itself and mate mapped
502150 + 0 singletons (6.66% : N/A)
65434 + 0 with mate mapped to a different chr
24127 + 0 with mate mapped to a different chr (mapQ>=5)

==== Sample_2 ====
7652717 + 0 in total (QC-passed reads + QC-failed reads)
7584878 + 0 primary
0 + 0 secondary
67839 + 0 supplementary
633723 + 0 duplicates
633723 + 0 primary duplicates
7049555 + 0 mapped (92.12% : N/A)
6981716 + 0 primary mapped (92.05% : N/A)
7584878 + 0 paired in sequencing
3792439 + 0 read1
3792439 + 0 read2
6113888 + 0 properly paired (80.61% : N/A)
6378554 + 0 with itself and mate mapped
603162 + 0 singletons (7.95% 

## 5. Somatic Variant Calling

In [17]:
# Running Mutect2 for each sample
raw_vcfs, orient_models = {}, {}
for run, name in SAMPLE_SHEET:
    bam   = os.path.join(BAM_DIR, f'{name}.recal.bam')
    raw   = os.path.join(VCF_DIR, f'{name}.unfiltered.vcf.gz')
    f1r2  = os.path.join(VCF_DIR, f'{name}.f1r2.tar.gz')
    model = os.path.join(VCF_DIR, f'{name}.read-orientation.tar.gz')
    print('Calling', name)
    os.system(
        f'gatk Mutect2 -R {REF_FA} -I {bam} '
        f'--germline-resource {GERMLINE} --panel-of-normals {PON} '
        f'--f1r2-tar-gz {f1r2} {INTERVALS} -O {raw}'
    )
    os.system(f'gatk LearnReadOrientationModel -I {f1r2} -O {model}')
    raw_vcfs[name], orient_models[name] = raw, model

Calling Sample_1
Calling Sample_2
Calling Sample_3


In [18]:
# Producing contamination tables and segments for each sample
contam_tables, segments = {}, {}
for run, name in SAMPLE_SHEET:
    bam      = os.path.join(BAM_DIR, f'{name}.recal.bam')
    pileup   = os.path.join(VCF_DIR, f'{name}.pileups.table')
    contam   = os.path.join(VCF_DIR, f'{name}.contamination.table')
    seg      = os.path.join(VCF_DIR, f'{name}.segments.table')
    print('Contamination for', name)
    os.system(f'gatk GetPileupSummaries -I {bam} -V {COMMON} -L {COMMON} -O {pileup}')
    os.system(f'gatk CalculateContamination -I {pileup} '
              f'--tumor-segmentation {seg} -O {contam}')
    contam_tables[name], segments[name] = contam, seg

Contamination for Sample_1
Contamination for Sample_2
Contamination for Sample_3


## 6. Variant Filtering

In [19]:
# Filtering variants with GATK4 FilterMutectCalls
filt_vcfs = {}
for run, name in SAMPLE_SHEET:
    raw    = os.path.join(VCF_DIR, f'{name}.unfiltered.vcf.gz')
    contam = os.path.join(VCF_DIR, f'{name}.contamination.table')
    seg    = os.path.join(VCF_DIR, f'{name}.segments.table')
    model  = os.path.join(VCF_DIR, f'{name}.read-orientation.tar.gz')
    filt   = os.path.join(VCF_DIR, f'{name}.filtered.vcf.gz')
    os.system(
        f'gatk FilterMutectCalls -R {REF_FA} -V {raw} '
        f'--contamination-table {contam} '
        f'--tumor-segmentation {seg} '
        f'--ob-priors {model} -O {filt}'
    )
    filt_vcfs[name] = filt
    n_pass = os.popen(f'bcftools view -f PASS -H {filt} | wc -l').read().strip()
    print(f'{name}: {n_pass} PASS candidate somatic variants')

Sample_1: 7951 PASS candidate somatic variants
Sample_2: 8027 PASS candidate somatic variants
Sample_3: 5257 PASS candidate somatic variants


## 7. Variant Annotation

In [24]:
# Annotating variants with snpEff
SNPEFF_DB = 'GRCh38.86'
if not os.path.isdir(f'snpEff/data/{SNPEFF_DB}'):
    print(f'Downloading snpEff DB {SNPEFF_DB} ...')
    os.system(f'java -jar snpEff/snpEff.jar download -v {SNPEFF_DB}')

os.system(f'bcftools index -f -t {GERMLINE}')

ann_tables, failed = {}, []
for run, name in SAMPLE_SHEET:
    filt     = os.path.join(VCF_DIR, f'{name}.filtered.vcf.gz')
    pass_vcf = os.path.join(VCF_DIR, f'{name}.pass.vcf.gz')
    af_vcf   = os.path.join(VCF_DIR, f'{name}.pass.af.vcf.gz')
    nochr    = os.path.join(VCF_DIR, f'{name}.nochr.vcf')
    ann_vcf  = os.path.join(ANN_DIR, f'{name}.snpeff.vcf')
    log      = os.path.join(ANN_DIR, f'{name}.snpeff.log')
    try:
        os.system(f'bcftools view -f PASS -O z -o {pass_vcf} {filt}')
        os.system(f'bcftools index -f -t {pass_vcf}')
        print('Annotating', name)
        os.system(f'bcftools annotate -a {GERMLINE} -c INFO/gnomAD_AF:=INFO/AF '
                  f'-O z -o {af_vcf} {pass_vcf}')
        os.system(f"zcat {af_vcf} | sed -e 's/^chr//' -e 's/ID=chr/ID=/' > {nochr}")
        ret = os.system(f'java -Xmx6g -jar snpEff/snpEff.jar -canon -noStats '
                        f'{SNPEFF_DB} {nochr} > {ann_vcf} 2> {log}')
        if ret == 0 and os.path.exists(ann_vcf) and os.path.getsize(ann_vcf) > 0:
            ann_tables[name] = ann_vcf
        else:
            failed.append(name)
            print(f'  ! snpEff failed for {name} — see {log}')
    except Exception as e:
        failed.append(name)
        print(f'  ! Annotation failed for {name}: {e}')

print(f'\n{len(ann_tables)} samples annotated successfully')
if failed:
    print('Skipped (annotation failed):', failed)

Annotating Sample_1
Annotating Sample_2
Annotating Sample_3

3 samples annotated successfully


## 8. Prioritisation

In [26]:
# Prioritizing variants
PANEL = ['BRCA1','BRCA2','TP53','PALB2','ATM','CHEK2','PTEN','CDH1','STK11','RAD51C','RAD51D','BARD1']
HIGH_MOD = ['missense_variant','frameshift_variant','stop_gained','stop_lost','start_lost',
            'splice_acceptor_variant','splice_donor_variant','inframe_insertion','inframe_deletion']

def load_snpeff(path):
    """Parse a snpEff-annotated VCF: gene, consequence, impact, protein, gnomAD AF."""
    if not (os.path.exists(path) and os.path.getsize(path) > 0):
        return pd.DataFrame()
    rows = []
    with open(path) as fh:
        for line in fh:
            if line.startswith('#'):
                continue
            f = line.rstrip('\n').split('\t')
            if len(f) < 8:
                continue
            chrom, pos, ref, alt, info = f[0], f[1], f[3], f[4], f[7]
            info_d = {}
            for kv in info.split(';'):
                if '=' in kv:
                    k, v = kv.split('=', 1); info_d[k] = v
            symbol = cons = impact = hgvsp = None
            ann = info_d.get('ANN')
            if ann:
                a = ann.split(',')[0].split('|')      # first (most severe) annotation
                cons   = a[1]  if len(a) > 1  else None   # consequence
                impact = a[2]  if len(a) > 2  else None   # HIGH/MODERATE/LOW/MODIFIER
                symbol = a[3]  if len(a) > 3  else None   # gene name
                hgvsp  = a[10] if len(a) > 10 else None   # HGVS.p (protein change)
            rows.append({
                'CHROM': chrom if chrom.startswith('chr') else 'chr'+chrom,
                'POS': pos, 'REF': ref, 'ALT': alt,
                'SYMBOL': symbol, 'Consequence': cons, 'IMPACT': impact,
                'HGVSp': hgvsp, 'gnomAD_AF': info_d.get('gnomAD_AF'),
            })
    return pd.DataFrame(rows)

def prioritise(df):
    if df.empty:
        return df
    df = df.copy()
    af = pd.to_numeric(df['gnomAD_AF'], errors='coerce')
    rare  = af.isna() | (af < 0.01)                              # rare/absent in gnomAD
    func  = df['Consequence'].fillna('').str.contains('|'.join(HIGH_MOD))
    panel = df['SYMBOL'].isin(PANEL)
    df['rare'], df['functional'], df['in_panel'] = rare, func, panel
    df['_ir'] = df['IMPACT'].map({'HIGH':3,'MODERATE':2,'LOW':1,'MODIFIER':0}).fillna(0)
    keep = (func & rare) | (panel & func)                       # no ClinVar branch (snpEff has none)
    out = df[keep].sort_values(['in_panel','_ir','functional'], ascending=False)
    return out.drop(columns='_ir')

all_hits = []
for run, name in SAMPLE_SHEET:
    ann_vcf = os.path.join(ANN_DIR, f'{name}.snpeff.vcf')
    df = load_snpeff(ann_vcf)
    if df.empty:
        print(f'{name}: no annotated variants, skipped'); continue
    pri = prioritise(df); pri.insert(0, 'sample', name)
    print(f'{name}: {len(pri)} candidate / {len(df)} annotated')
    all_hits.append(pri)

candidates = pd.concat(all_hits, ignore_index=True) if all_hits else pd.DataFrame()
out_csv = os.path.join(RESULTS_DIR, 'candidate_somatic_variants.csv')
candidates.to_csv(out_csv, index=False)
print('\nSaved', out_csv)
candidates.head(20)

Sample_1: 174 candidate / 7951 annotated
Sample_2: 169 candidate / 8027 annotated
Sample_3: 73 candidate / 5257 annotated

Saved results/candidate_somatic_variants.csv


,sample,CHROM,POS,REF,ALT,SYMBOL,Consequence,IMPACT,HGVSp,gnomAD_AF,rare,functional,in_panel
0,Sample_1,chr17,7670664,C,A,TP53,stop_gained,HIGH,p.Glu310*,None,True,True,True
1,Sample_1,chr13,46715621,C,CCCTCAAGGCCCCCAACAGGAACCCACCTAATGATCACACAGTCCA...,LRCH1,stop_gained&conservative_inframe_insertion,HIGH,p.Asp572_Ser573insProGlnGlyProGlnGlnGluProThrT...,None,True,True,False
2,Sample_1,chr13,67227063,C,CT,PCDH9,frameshift_variant,HIGH,p.Glu460fs,None,True,True,False
3,Sample_1,chr13,94601960,C,CTGGACAGGAGGGACCATCCACTCCATAGCCCTCACCTCCCTTACC...,GPR180,frameshift_variant,HIGH,p.Thr12fs,None,True,True,False
4,Sample_1,chr13,109100811,T,TCTCATATCAATCTGCTGGTAGTTAGAAGGATTCATGATCAAATTT...,MYO16,frameshift_variant&stop_gained,HIGH,p.Arg1122fs,None,True,True,False
5,Sample_1,chr13,110173978,G,GAGCTGCTCTGTTGTCTGCGTCATCGAGATCATCGAGGTCTTCTTC...,COL4A1,frameshift_variant&stop_gained,HIGH,p.Pro1143fs,None,True,True,False
6,Sample_1,chr13,113633988,GA,AG,TFDP1,stop_gained,HIGH,p.TrpIle191*,None,True,True,False
7,Sample_1,chr13,113863704,C,CCTGGGCATGGTCTACCTCCGGATCGGGTAAGGGCTGCCTGCGCCC...,GAS6,stop_gained&disruptive_inframe_insertion,HIGH,p.Leu42_Arg43insCysArgAsnThrArgGlyLysGlyAlaGly...,None,True,True,False
8,Sample_1,chr17,2371363,T,TCCTCCGGCTGACCTCGGGGGCTGGCTTGCTG,SGSM2,frameshift_variant&stop_gained,HIGH,p.Ser511fs,None,True,True,False
9,Sample_1,chr17,2371367,G,GGGCTGGGGAGCCACCCGGGCGTGTGCCCCAAC,SGSM2,frameshift_variant,HIGH,p.Ser510fs,None,True,True,False


## 9. Interpretation & Report

In [29]:
# Generating a simple report of the analysis
out_csv = os.path.join(RESULTS_DIR, 'candidate_somatic_variants.csv')
candidates = pd.read_csv(out_csv)

lines = ['# Tumour-Only Somatic Analysis — Candidate Variant Report', '',
         '**Dataset:** SRP676624 / PRJNA1422845 (TNBC whole-exome, tumour-only)',
         f'**Samples:** {len(SAMPLE_SHEET)}',
         f'**Reference subset:** {", ".join(CHROMS)}',
         '**Annotation:** snpEff (GRCh38.86)', '',
         '_Candidates are candidate somatic variants: the tumour-only design cannot exclude '
         'rare germline variants without a matched normal._', '']

if len(candidates):
    panel_hits = candidates[candidates['in_panel']]
    lines += [f'## Cancer-panel candidates ({len(panel_hits)})', '']
    if len(panel_hits):
        for _, r in panel_hits.iterrows():
            af = r.get('gnomAD_AF')
            af_txt = 'absent from gnomAD' if pd.isna(af) else f'gnomAD AF {af}'
            lines.append(f"- **{r['SYMBOL']}** ({r['sample']}): {r.get('Consequence','?')}, "
                         f"{r.get('HGVSp','')} — {r.get('IMPACT','')} impact, {af_txt}")
        rec = panel_hits.groupby('SYMBOL')['sample'].nunique()
        rec = rec[rec > 1]
        if len(rec):
            lines += ['', '## Recurrent panel genes (in >1 sample)', '']
            for g, n in rec.items():
                lines.append(f'- {g}: {n} samples')
    else:
        lines.append('No panel-gene candidates passed the filters.')

    # per-sample accounting so all samples appear, even with zero panel hits
    lines += ['', '## Per-sample summary', '']
    for run, name in SAMPLE_SHEET:
        n_total = (candidates['sample'] == name).sum()
        n_panel = ((candidates['sample'] == name) & candidates['in_panel']).sum()
        lines.append(f'- **{name}**: {n_total} candidate variants, {n_panel} in cancer panel')

    lines += ['', f'## Total candidate variants: {len(candidates)}',
              '_Most non-panel candidates are novel/uncharacterised and include alignment '
              'artifacts inherent to the two-chromosome subset reference; the cancer-panel '
              'hits above are the biologically interpretable findings._']
else:
    lines.append('No candidates (check the pipeline ran completely).')

report = os.path.join(RESULTS_DIR, 'analysis_report.md')
open(report, 'w').write('\n'.join(lines))
print('\n'.join(lines))
print('\nReport saved to', report)

# Tumour-Only Somatic Analysis — Candidate Variant Report

**Dataset:** SRP676624 / PRJNA1422845 (TNBC whole-exome, tumour-only)
**Samples:** 3
**Reference subset:** chr13, chr17
**Annotation:** snpEff (GRCh38.86)

_Candidates are candidate somatic variants: the tumour-only design cannot exclude rare germline variants without a matched normal._

## Cancer-panel candidates (2)

- **TP53** (Sample_1): stop_gained, p.Glu310* — HIGH impact, absent from gnomAD
- **TP53** (Sample_2): frameshift_variant, p.Gly69fs — HIGH impact, absent from gnomAD

## Recurrent panel genes (in >1 sample)

- TP53: 2 samples

## Per-sample summary

- **Sample_1**: 174 candidate variants, 1 in cancer panel
- **Sample_2**: 169 candidate variants, 1 in cancer panel
- **Sample_3**: 73 candidate variants, 0 in cancer panel

## Total candidate variants: 416
_Most non-panel candidates are novel/uncharacterised and include alignment artifacts inherent to the two-chromosome subset reference; the cancer-panel hits above 

## 10. Session Info

- Colab CPU 2026.04 Python 3 (ipykernel) run in Visual Studio Code x64 1.120.0